<a href="https://colab.research.google.com/github/GodesAleksandra/DI-Bootcamp/blob/main/Daily_Challenge_MCP_Weather_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Daily Challenge: MCP Weather (Student)
Beginner-friendly MCP server + client (no LLMs). Complete the TODOs, then run the client.

## Setup
Run the install cell. If Colab asks, restart runtime after install.

In [9]:
# Install MCP CLI + SDK
%pip install -qU "mcp[cli]"

In [10]:
# Quick verify
!python --version
!mcp --help | head -n 5

Python 3.13.16
                                                                                
 Usage: mcp [OPTIONS] COMMAND [ARGS]...                                         
                                                                                
 MCP development tools                                                          
                                                                                


## A) Server (server.py)
Implement the WeatherDemo server with one tool and one resource.

In [11]:
%%writefile server.py
import logging
from mcp.server.mcpserver import MCPServer, Context
from mcp.types import CallToolResult, TextContent

# Configure logging
logging.basicConfig(level=logging.INFO)

# Initialize MCPServer (v2 architecture)
mcp = MCPServer("WeatherDemo")

CITY_DATA = {
    "paris": {"temp_c": 21, "condition": "sunny"},
    "london": {"temp_c": 18, "condition": "cloudy"},
    "new york": {"temp_c": 24, "condition": "breezy"},
}

@mcp.tool()
def get_weather(city: str, ctx: Context) -> CallToolResult:
    """Return simple weather info for a supported city."""
    key = city.strip().lower()
    data = CITY_DATA.get(key)

    if not data:
        return CallToolResult(
            content=[TextContent(type="text", text=f"Unsupported city: {city}. Try one of: {', '.join(CITY_DATA)}")],
            is_error=True
        )

    logging.info("get_weather called for %s", key)

    return CallToolResult(
        content=[TextContent(type="text", text=f"The weather in {city} is {data['temp_c']}°C and {data['condition']}.")]
    )

# FIXED: Removed 'ctx: Context' because 'cities://list' has no template parameters
@mcp.resource("cities://list")
def list_cities() -> str:
    """Return supported cities as newline-separated text."""
    return "\n".join(sorted(CITY_DATA))

if __name__ == "__main__":
    mcp.run()


Overwriting server.py


## B) Client (client.py)
Spawn the server via STDIO, discover capabilities, and call them.

In [12]:
%%writefile client.py
import asyncio
import sys
from pathlib import Path
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# Define local server location
SERVER_PATH = Path(__file__).parent / "server.py"
server_params = StdioServerParameters(
    command=sys.executable, args=[str(SERVER_PATH)], env=None
)

def extract_content(payload):
    """Safely extracts text blocks from v2 ReadResourceResult or CallToolResult."""
    # Check for Resource Content array
    if hasattr(payload, "contents") and payload.contents:
        first = payload.contents[0]
        if hasattr(first, "text"):
            return first.text
        return str(first)

    # Check for Tool Content array
    if hasattr(payload, "content") and payload.content:
        first = payload.content[0]
        if hasattr(first, "text"):
            return first.text
        return str(first)

    return str(payload)

async def run():
    print("Connecting to MCP v2 Server...")
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            print("Session successfully initialized!\n")

            # 1. Verify Resources
            resources = await session.list_resources()
            print("Available Resources:")
            for res in resources.resources:
                print(f"- {res.uri} ({res.name or ''})")

            # 2. Verify Tools
            tools = await session.list_tools()
            print("\nAvailable Tools:")
            for tool in tools.tools:
                print(f"- {tool.name}")

            # 3. Read Static Resource
            cities = await session.read_resource("cities://list")
            print("\nReading 'cities://list' resource contents:")
            print(extract_content(cities))

            # 4. Call Tool
            weather = await session.call_tool("get_weather", {"city": "Paris"})
            print("\nExecuting 'get_weather' tool for Paris:")
            print(extract_content(weather))

if __name__ == "__main__":
    asyncio.run(run())


Overwriting client.py


## C) Run
Single terminal (spawns server):
```
python client.py
```
Or two terminals for debugging:
```
mcp run server.py
python client.py
```
In Colab, just run the next cell.

In [13]:
!python client.py

Connecting to MCP v2 Server...
Session successfully initialized!

Available Resources:
- cities://list (list_cities)

Available Tools:
- get_weather

Reading 'cities://list' resource contents:
london
new york
paris
INFO:root:get_weather called for paris

Executing 'get_weather' tool for Paris:
The weather in Paris is 21°C and sunny.


## Troubleshooting
- `mcp: command not found` ? rerun install or restart runtime.
- No tools/resources ? check decorators, restart server.
- City missing ? use one of the supported cities listed by `cities://list`.